# Citi Velocity USTs — EOD

Citi Velocity as the truth source for US Treasuries, through the usual
**MDP → TimeseriesBuilder → Query** path.

Two things make this source different from the other UST sources in the repo:

- **It is a quote service.** Citi publishes **46** values per bond on this tag path
  and no pricer is entitled, so anything Citi does not publish is rebuilt locally
  in rateslib / QuantLib from the quote it does publish. Which of the two you got
  is recorded on every number — see the provenance section below.
- **It carries a liquid subset, not every UST.** 349 nominal coupon bonds,
  0.02y to 29.8y. No bills, no TIPS, no FRNs, no STRIPS. And it lags new auctions
  — see the caveat at the bottom, which is the thing most likely to surprise you.

The whole universe is warmed nightly by
`scripts/citivelo_ust_universe_warm.py`, so everything here reads from cache and
opens no workbook.

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

import pandas as pd
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue

# "-RL" builds rateslib pricers, "-QL" QuantLib. Both read the same Citi quotes.
usts_mdp = FixedRateBondsMDP(source="USTS_CITIVELO-RL")
ts_builder = TimeseriesBuilder()

## The universe

`BondUniverse` reads the committed catalog — no Excel, no network. Coverage is
**per bond**: every bond has `PRICE`/`YIELD`/`DURATION`/`SPREAD_TSY`, but only
305 of 349 serve `DV01` and 253 serve `ASW_4_USD`. Ask
`available_values(isin)` rather than assuming.

In [3]:
from MDP.CitiVelocityExcel.bonds import BondUniverse

uni = BondUniverse.from_catalog(country="USA", asset_type="GOVT")
print(f"{len(uni)} bonds, {uni.to_frame()['maturity'].min()} .. {uni.to_frame()['maturity'].max()}")

frame = uni.to_frame()
display(frame.head())

# The measured vocabulary. 46 values serve on this tag path; a single US
# Treasury serves 44. The earlier answer of 8 came from a probe that used a
# ONE-WEEK window against one bond, so "no rows in a week" was recorded as
# "does not exist" — which lost ZSPREAD, ASW, CAS and three ASW legs.
from MDP.CitiVelocityExcel import tags as T

print(f"{len(T.BOND_VALUES)} values in the vocabulary")

import collections
cov = collections.Counter(v for d in uni for v in uni.available_values(d.isin))
pd.Series(cov).sort_values(ascending=False).to_frame("bonds serving").head(20)

349 bonds, 2026-08-15 .. 2056-05-15


,isin,ticker,coupon,maturity,country,currency,asset_type,description,source,priceable
0,US912810EX29,T,6.750,2026-08-15,USA,USD,GOVT,T 6.75 08/15/2026,catalog,True
1,US9128282A70,T,1.500,2026-08-15,USA,USD,GOVT,T 1.5 08/15/2026,catalog,True
2,US91282CHU80,T,4.500,2026-08-15,USA,USD,GOVT,T 4.5 08/15/2026,catalog,True
3,US912828YD60,T,1.375,2026-08-31,USA,USD,GOVT,T 1.375 08/31/2026,catalog,True
4,US91282CCW91,T,0.750,2026-08-31,USA,USD,GOVT,T 0.75 08/31/2026,catalog,True


46 values in the vocabulary


,bonds serving
DURATION,349
DV01,349
OAS,349
PRICE,349
YIELD,349
ASW_4_EUR,349
ASW_4_CHF,349
SPREAD_TSY,349
ASW_4_GBP,349
ASW_4_JPY,349


### The families guesswork missed

Probing Citi's own 118-field dictionary surfaced whole families that no amount of
guessing at names would have produced:

| family | values |
|---|---|
| carry / roll | `CARRY.1M/3M/6M/1Y`, `ROLL.*`, `ROLLCARRY.*` |
| asset-swap variants | `ASW`, `ASW_RFR`, `ASWNP`, `ASSNP_RFR`, `ASS_SOFR` |
| OIS spread | `OISS`, `OISS_RFR`, `OISSMM`, `OISSMM_RFR` |
| yield-yield spread | `YYS`, `YYS_RFR`, `YYS_SOFR` |
| coupon-adjusted spread | `CAS`, `CAS_RFR`, `CAS_SOFR` |
| pricing | `PRICING_ACCRUED`, `PRICING_CV01` |

**Two traps.** Six values — `ASW`, `ASWNP`, `CAS`, `OISS`, `YYS`, `ZSPREAD` — were
**retired on 2025-10-03** when Citi moved the family to an RFR basis. They still
return ~1,039 rows, so they look served; only a recent date fails, and it fails as
"no rows in the window". Use the `_RFR` successor.

And **carry/roll lag by exactly their horizon** (`CARRY.6M` ends ~6 months back),
which says they are realised over the window just ended, not forecast.

In [4]:
from MDP.CitiVelocityExcel.bonds import values as V

pd.Series(V.DISCONTINUED_2025_10_03, name="use instead").to_frame()

,use instead
ASW,ASW_RFR
ASWNP,ASSNP_RFR
CAS,CAS_RFR
OISS,OISS_RFR
YYS,YYS_RFR
ZSPREAD,


## Outright values

`UnifiedQuery(cusip=...)` takes an on-the-run alias (`CT10`, `O5`, `OO2`) or a
literal CUSIP. The alias is resolved against the UST reference table first, then
the CUSIP is converted to Citi's ISIN — `US` + CUSIP + ISO 6166 check digit,
verified against all 2,162 ISINs Citi serves.

In [5]:
start = datetime.date(2026, 7, 20)
end   = datetime.date(2026, 8, 7)

queries = [
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_YTM),            # computed here
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_CITI_YIELD),     # Citi's own
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_SPREAD_TSY),     # G-spread
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_ASW_RFR),        # ASW vs RFR
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_CAS_RFR),        # CAS vs RFR
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_OAS_RFR),        # OAS vs RFR
]

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=queries,
    routers={
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    ignore_cache_miss=True,
)
df

DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb
PRICING FIXED-RATE BONDS.:   0%|          | 0/90 [00:00<?, ?it/s]Pricing failed for cusip='CT10', date='2026-07-20', query='FixedRateBondQuery(product='FRB', structure_id=<FixedRateBondStructure.OUTRIGHT: 1>, structure_kwargs={'cusip': 'CT10'}, value_id=<FixedRateBondValue.CAS_RFR: 32>, value_ids=(), market_request={}, mdp_time_key='timestamp', name=None, tags=(), meta={}, structure=<FixedRateBondStructure.OUTRIGHT: 1>, value=<FixedRateBondValue.CAS_RFR: 32>, cusip='CT10', curve=None, value_kwargs={}, risk_weight=None)'. Error: Citi does not serve CAS_RFR for CT10, so it was never requested. Per-bond coverage is uneven and is read from the harvested validation set; this bond serves PRICE, YIELD, SPREAD_TSY, OAS, DURATION, DV01, ASW_4_USD, ASW_4_JPY, ASW_4_AUD.
Traceback (most recent call last):
  File "c:\Users\chris\clee\ARBS\notebooks\timeseries\../..\TB\FixedRateBondsT

,CT10 OUTRIGHT CITI_YIELD,CT10 OUTRIGHT SPREAD_TSY,CT10 OUTRIGHT YTM
Date,,,
2026-07-20,4.59575,-0.953521,4.594990
2026-07-21,4.62608,-0.907164,4.625297
2026-07-22,4.65651,-0.892520,4.655717
2026-07-23,4.70232,-1.035740,4.701505
2026-07-24,4.67707,-1.043900,4.676253
2026-07-27,4.64007,-1.081080,4.639256
2026-07-28,4.60318,-1.060860,4.602385
2026-07-29,4.62042,-1.095270,4.619616
2026-07-30,4.66105,-1.294780,4.660225


Not every column is guaranteed: the full 46-value coverage sweep has been
run for 120 of the 349 US Treasuries so far (it stops at the Excel memory ceiling
and resumes), so a bond outside that set still carries only the values the
earlier sweep validated. Missing columns here mean "not probed yet for this
bond", not "Citi does not serve it" — `citivelo_ust_universe_warm.py` finishes
the job after an Excel restart.

In [6]:
served = [q.col_name() for q in queries if q.col_name() in df.columns]
missing = [q.col_name() for q in queries if q.col_name() not in df.columns]
print(f"served: {served}")
if missing:
    print(f"not yet probed for this bond: {missing}")

plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(df[served[0]], which="left")
if len(served) > 1:
    plot(df[served[-1]], which="right")
legend(valfmt="{:.4f}", show_date=True)

served: ['CT10 OUTRIGHT YTM', 'CT10 OUTRIGHT CITI_YIELD', 'CT10 OUTRIGHT SPREAD_TSY']
not yet probed for this bond: ['CT10 OUTRIGHT ASW_RFR', 'CT10 OUTRIGHT CAS_RFR', 'CT10 OUTRIGHT OAS_RFR']


## Quoted or computed? — provenance

`FRB_SPREAD_TSY`, `FRB_OAS`, `FRB_ASW_SPREAD` and `FRB_CAS` are **quote-only**:
they return Citi's published number and raise on a pricer from any other source,
rather than returning `0.0`, which is a perfectly plausible spread.

Everything else is **computed locally** from Citi's `PRICE`. The pricer records
which, so a surprising number is traceable instead of needing to be re-derived.

The four readings that could have been confidently wrong were settled by
measurement, not by the tag name — by asking which reading of one Citi number
reproduces another:

| value | verdict | margin |
|---|---|---|
| `PRICE` | **clean**, per 100 | 0.0186 bp vs Citi's own YIELD; **57.43 bp** if read as dirty |
| `DURATION` | **modified** | 1.9e-05 yr; **0.0500 yr** if read as Macaulay |
| `DV01` | **per 1mm face**, + for a long | ratio to local per-100 dv01 = 10000.06 |

In [7]:
from MDP.CitiVelocityExcel.bonds import values as V

pricers = usts_mdp.get_data({"cusips": ["CT10"], "timestamp": end})
meta = pricers["CT10"].meta()

rows = []
for name in ("CLEAN_PRICE", "YTM", "MOD_DURATION", "DV01", "SPREAD_TSY"):
    p = V.provenance_of(meta, name)
    if p is not None:
        rows.append({"value": name, "origin": p.origin, "unit": p.unit,
                     "verified": p.verified, "detail": p.detail})
display(pd.DataFrame(rows))

# What Citi published for this bond, verbatim and unscaled:
pd.Series(V.quoted_value_of(meta, v) for v in ("PRICE", "YIELD", "DURATION", "DV01")), \
    V.coverage_of(meta)

,value,origin,unit,verified,detail
0,CLEAN_PRICE,quoted,price_points,True,RATES.BOND.US91282CQQ77.PRICE
1,YTM,computed,percent,True,rateslib.RLFixedRateBondPricer.ytm<-RATES.BOND...
2,MOD_DURATION,computed,years,True,rateslib.RLFixedRateBondPricer.mod_duration<-R...
3,DV01,computed,currency_per_bp,True,rateslib.RLFixedRateBondPricer.dv01<-RATES.BON...
4,SPREAD_TSY,quoted,basis_points,False,RATES.BOND.US91282CQQ77.SPREAD_TSY


(0     97.79690
 1      4.65759
 2      7.76356
 3    767.28200
 dtype: float64,
 {'serves': ('DURATION',
   'DV01',
   'OAS',
   'PRICE',
   'YIELD',
   'SPREAD_TSY',
   'ASW_4_CHF',
   'ASW_4_EUR',
   'ASW_4_GBP',
   'ASW_4_USD',
   'ASW_4_AUD',
   'ASW_4_JPY'),
  'requested': ('DURATION',
   'DV01',
   'OAS',
   'PRICE',
   'YIELD',
   'SPREAD_TSY',
   'ASW_4_CHF',
   'ASW_4_EUR',
   'ASW_4_GBP',
   'ASW_4_USD',
   'ASW_4_AUD',
   'ASW_4_JPY'),
  'unavailable': ('ASSNP_RFR',
   'ASW_RFR',
   'CARRY.1M',
   'CARRY.1Y',
   'CARRY.3M',
   'CARRY.6M',
   'PRICING_ACCRUED',
   'PRICING_CV01',
   'CAS',
   'CAS_RFR',
   'OISSMM_RFR',
   'OISS_RFR',
   'YYS',
   'YYS_RFR',
   'ZSPREAD',
   'ASW',
   'ASWNP',
   'OAS_RFR',
   'OISS',
   'OISSMM',
   'ROLL.1M',
   'ROLL.1Y',
   'ROLL.3M',
   'ROLL.6M',
   'ROLLCARRY.1M',
   'ROLLCARRY.1Y',
   'ROLLCARRY.3M',
   'ROLLCARRY.6M',
   'OISS_SOFR',
   'ASS_SOFR',
   'CAS_SOFR',
   'SIMPLEYIELD',
   'YIELD_WORST',
   'YYS_SOFR'),
  'empty': ('ASW_4

## Curves and flies

Two legs make a `CURVE`, three a `FLY`. The `/` spelling is the same one the
other UST notebooks use.

In [8]:
curve_q = UnifiedQuery(cusip="CT5/CT10", value=UnifiedValue.FRB_YTM)
fly_q   = UnifiedQuery(cusip="CT2/CT5/CT10", value=UnifiedValue.FRB_YTM)

curves = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[curve_q, fly_q],
    routers={"FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True)},
    n_jobs=12,
    ignore_cache_miss=True,
)
curves

WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb
FETCHING PRICERS:   9%|▉         | 1/11 [00:00<00:04,  2.37it/s]WARNING	Thread(frb-mdp_10) MDP.FixedRateBonds.FixedRateBondsMDP:FixedRateBondsMDP.py:_get_multi_pricers()- citivelo bonds: 91282CRA1 not resolvable to a quoted bond: BondNotQuotedError: '91282CRA1' resolves to US91282CRA17 via cusip, but Citi does not quote it. The arithmetic is not in question - it reproduces all 2,162 ISINs Citi serves - so this is a coverage gap: Citi carries a liquid subset (349 US Treasuries), not every issue. Off-the-runs, bills, TIPS and just-auctioned bonds are the usual misses.
WARNING	Thread(frb-mdp_10) MDP.FixedRateBonds.FixedRateBondsMDP:FixedRateBondsMDP.py:_get_multi_pricers()- citivelo bonds: 91282CRB9 not resolvable to a quoted bond: BondNotQuotedError: '91282CRB9' reso

,CT2/CT5/CT10 FLY YTM,CT5/CT10 CURVE YTM
Date,,
2026-07-20,-15.118522,26.628575
2026-07-21,-15.024548,25.817067
2026-07-22,-14.327531,24.916116
2026-07-23,-13.567612,24.103644
2026-07-24,-15.458608,25.090338
2026-07-27,-16.667043,24.476712
2026-07-28,-15.906941,24.547677
2026-07-29,-15.444030,27.223703
2026-07-30,-14.081763,28.817286


## Caveat — Citi lags new auctions

Measured 2026-08-08 against the repo's own `fiscaldata` reference table: **352**
live nominal coupon USTs, of which Citi carried **349**. The six absent were
three when-issued (settling 2026-08-17, correctly not quoted) and **three issued
2026-07-31 — eight days earlier — that Citi had still not picked up.**

Those three are the on-the-run 2Y, 5Y and 7Y, so `UnifiedQuery(cusip="CT2")`
resolves to a real bond that this source cannot quote. Resolution raises a
coverage error naming it rather than returning nothing, so it is loud — but it
means **the freshest on-the-run may not be available here**. Use the previous
issue (`O2`) or another source when you need the very newest bond.

`scripts/citivelo_ust_universe_warm.py refresh` picks up new bonds as soon as
Citi has them, and never removes: matured bonds are unrecoverable from
`CVCURVEBOND` (asking it for an old date returns today's set filtered, not the
set as it stood), so anything seen once is kept and its cached history stays
valid.

In [9]:
from MDP.CitiVelocityExcel.bonds.resolution import resolve_bond

# resolve_bond takes a CUSIP or an ISIN. On-the-run ALIASES (CT10, O2) are
# resolved to a CUSIP first, by FixedRateBondsMDP against the UST reference
# table -- there is one alias table in this repo, not a second one here.
for cusip, what in [
    ("91282CNJ6", "a bond Citi carries"),
    ("91282CRA1", "the 5Y issued 2026-07-31 -- Citi has not picked it up"),
    ("037833100", "Apple: a real ISIN, valid check digit, no rates desk quotes it"),
]:
    try:
        r = resolve_bond(cusip)
        print(f"{cusip:<12} {what:<52} -> {r.isin}  {r.descriptor.description}")
    except Exception as e:
        print(f"{cusip:<12} {what:<52} -> {type(e).__name__}")
        print(f"{'':<12} {str(e)[:150]}")

91282CNJ6    a bond Citi carries                                  -> US91282CNJ61  T 4.0 06/30/2032
91282CRA1    the 5Y issued 2026-07-31 -- Citi has not picked it up -> BondNotQuotedError
             '91282CRA1' resolves to US91282CRA17 via cusip, but Citi does not quote it. The arithmetic is not in question - it reproduces all 2,162 ISINs Citi ser
037833100    Apple: a real ISIN, valid check digit, no rates desk quotes it -> BondNotQuotedError
             '037833100' resolves to US0378331005 via cusip, but Citi does not quote it. The arithmetic is not in question - it reproduces all 2,162 ISINs Citi ser
